# Tutorial 4: Molecular Descriptors with ASE

**Time**: 20 minutes

**Topics**:
- ASE database creation
- Molecular descriptor extraction
- Exact conditional generation
- SMARTS patterns

In [ ]:
import sys; sys.path.insert(0, '..')
import torch
import numpy as np
from ase import Atoms
from ase.db import connect
from qm9 import dataset
from qm9.utils import extract_molecular_descriptors

print('ASE Molecular Descriptors Tutorial')

## 1. Create ASE Database

In [ ]:
# Create database
db = connect('descriptors.db', append=False)

# Add molecules with descriptors
molecules = [
    {'atoms': 'C2H6', 'pos': [[0,0,0], [1.5,0,0]], 'mw': 30.0},
    {'atoms': 'C6H6', 'pos': [[0,1.4,0], [1.21,0.7,0]], 'mw': 78.0},
]

for mol in molecules:
    atoms = Atoms(mol['atoms'], positions=mol['pos'])
    db.write(atoms, data={'molecular_weight': mol['mw']})

print(f'Created database with {len(molecules)} molecules')

## 2. Extract Descriptors

**Available**:
- molecular_weight
- pi_conjugation_ratio
- atom_types_encoding
- functional_groups_encoding

In [ ]:
# Load and examine
for row in db.select():
    atoms = row.toatoms()
    print(f'\nMolecule: {atoms.get_chemical_formula()}')
    print(f'  Positions: {atoms.positions.shape}')
    print(f'  MW: {row.data.molecular_weight}')
    
    # Compute descriptors
    desc = extract_molecular_descriptors(atoms)
    print(f'  Descriptors: {desc.keys()}')

## 3. Train with Descriptors

```bash
python main_qm9.py \\
    --dataset ase_db \\
    --ase_db_path descriptors.db \\
    --conditioning molecular_weight pi_conjugation_ratio \\
    --exp_name descriptor_model
```

## 4. Exact Conditional Generation

```bash
python eval_conditional_qm9.py \\
    --generators_path outputs/descriptor_model \\
    --use_exact_conditions \\
    --property_values 'molecular_weight=50.0,pi_conjugation_ratio=0.9'
```

## Summary

✅ ASE database creation
✅ Descriptor extraction
✅ Training with descriptors
✅ Exact generation

Next: Tutorial 5 - Evaluation and Analysis